# Lab 3: Contextual Bandit-Based News Article Recommendation

**`Course`:** Reinforcement Learning Fundamentals  
**`Student Name`:**  
**`Roll Number`:**  
**`GitHub Branch`:** firstname_U20230xxx  

# Imports and Setup

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from rlcmab_sampler import sampler

# Load Datasets

In [4]:
# Load datasets
news_df = pd.read_csv("data/news_articles.csv")
train_users = pd.read_csv("data/train_users.csv")
test_users = pd.read_csv("data/test_users.csv")

print(news_df.head())
print(train_users.head())


                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla K. Johnson, AP   
1  He was subdued by passengers and crew when he ...        Mary Papenfuss   
2  "Until you have a dog y

## Data Preprocessing

In this section:
- Handle missing values
- Encode categorical features
- Prepare data for user classification

In [6]:
# Explore the datasets
print("News Articles Dataset Shape:", news_df.shape)
print("\nNews Articles Info:")
print(news_df.info())
print("\nNews Categories:", news_df['category'].unique())
print("\nNews Category Distribution:")
print(news_df['category'].value_counts())

print("\n" + "="*50)
print("\nTrain Users Dataset Shape:", train_users.shape)
print("\nTrain Users Info:")
print(train_users.info())
print("\nUser Categories:", train_users.iloc[:, -1].unique())
print("\nUser Category Distribution:")
print(train_users.iloc[:, -1].value_counts())

print("\n" + "="*50)
print("\nTest Users Dataset Shape:", test_users.shape)
print("\nTest Users Info:")
print(test_users.info())


News Articles Dataset Shape: (209527, 6)

News Articles Info:
<class 'pandas.DataFrame'>
RangeIndex: 209527 entries, 0 to 209526
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   link               209527 non-null  str  
 1   headline           209521 non-null  str  
 2   category           209527 non-null  str  
 3   short_description  189815 non-null  str  
 4   authors            172109 non-null  str  
 5   date               209527 non-null  str  
dtypes: str(6)
memory usage: 9.6 MB
None

News Categories: <StringArray>
['Crime', 'Entertainment', 'Education', 'Tech']
Length: 4, dtype: str

News Category Distribution:
category
Entertainment    154371
Education         42121
Tech               8096
Crime              4939
Name: count, dtype: int64


Train Users Dataset Shape: (2000, 33)

Train Users Info:
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 33 columns):
 #   

In [7]:
# Check missing values in detail
print("Missing values in news_df:")
print(news_df.isnull().sum())
print("\nMissing values in train_users:")
print(train_users.isnull().sum())
print("\nMissing values in test_users:")
print(test_users.isnull().sum())

# For news articles, we only need the category column for recommendation
# Fill missing values in headline if any
news_df['headline'] = news_df['headline'].fillna('Unknown')
news_df['short_description'] = news_df['short_description'].fillna('Unknown')
news_df['authors'] = news_df['authors'].fillna('Unknown')

# Standardize category names to match the problem statement
category_mapping = {
    'Crime': 'Crime',
    'Entertainment': 'Entertainment',
    'Education': 'Education',
    'Tech': 'Tech'
}
news_df['category'] = news_df['category'].map(category_mapping)

print("\nNews data after preprocessing:")
print(news_df['category'].value_counts())
print("\nNews data shape:", news_df.shape)


Missing values in news_df:
link                     0
headline                 6
category                 0
short_description    19712
authors              37418
date                     0
dtype: int64

Missing values in train_users:
user_id                        0
age                            0
income                         0
clicks                         0
purchase_amount                0
session_duration               0
content_variety                0
engagement_score               0
num_transactions               0
avg_monthly_spend              0
avg_cart_value                 0
browsing_depth                 0
revisit_rate                   0
scroll_activity                0
time_on_site                   0
interaction_count              0
preferred_price_range          0
discount_usage_rate            0
wishlist_size                  0
product_views                  0
repeat_purchase_gap (days)     0
churn_risk_score               0
loyalty_index                  0
screen_

## User Classification

Train a classifier to predict the user category (`User1`, `User2`, `User3`),
which serves as the **context** for the contextual bandit.


In [9]:
# Separate features and labels for train and test users
X_train_full = train_users.iloc[:, :-1].copy()
y_train_full = train_users.iloc[:, -1].copy()
X_test_final = test_users.copy()

print("Train features shape:", X_train_full.shape)
print("Train labels shape:", y_train_full.shape)
print("Test features shape:", X_test_final.shape)

print("\nUser category distribution in training data:")
print(y_train_full.value_counts())

# Identify categorical and numerical columns
categorical_cols = X_train_full.select_dtypes(include=['object', 'str']).columns.tolist()
numerical_cols = X_train_full.select_dtypes(include=['int64', 'float64', 'bool']).columns.tolist()

print(f"\nCategorical columns: {categorical_cols}")
print(f"Numerical columns: {len(numerical_cols)} columns")

# Label encode categorical columns - fit on combined data to handle unseen categories
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    # Combine train and test values to fit encoder
    combined_values = pd.concat([X_train_full[col], X_test_final[col]]).astype(str)
    le.fit(combined_values)
    
    # Transform both datasets
    X_train_full[col] = le.transform(X_train_full[col].astype(str))
    X_test_final[col] = le.transform(X_test_final[col].astype(str))
    label_encoders[col] = le

# Convert boolean to int
if 'subscriber' in X_train_full.columns:
    X_train_full['subscriber'] = X_train_full['subscriber'].astype(int)
    X_test_final['subscriber'] = X_test_final['subscriber'].astype(int)

print("\nData preprocessing completed!")
print("Train features shape after encoding:", X_train_full.shape)
print("Test features shape after encoding:", X_test_final.shape)


Train features shape: (2000, 32)
Train labels shape: (2000,)
Test features shape: (2000, 32)

User category distribution in training data:
label
user_2    712
user_1    707
user_3    581
Name: count, dtype: int64

Categorical columns: ['browser_version', 'region_code']
Numerical columns: 30 columns

Data preprocessing completed!
Train features shape after encoding: (2000, 32)
Test features shape after encoding: (2000, 32)


In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# Split train data into train and validation sets (80-20 split)
# Use roll number as random state for reproducibility
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_train_full
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")
print(f"\nTraining set label distribution:")
print(y_train.value_counts())
print(f"\nValidation set label distribution:")
print(y_val.value_counts())

# Train a Decision Tree classifier
user_classifier = DecisionTreeClassifier(random_state=42, max_depth=10)
user_classifier.fit(X_train, y_train)

# Predictions on validation set
y_val_pred = user_classifier.predict(X_val)

# Evaluate the classifier
val_accuracy = accuracy_score(y_val, y_val_pred)
print(f"\n{'='*60}")
print(f"User Classification Model Performance")
print(f"{'='*60}")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"\nClassification Report on Validation Set:")
print(classification_report(y_val, y_val_pred))

# Train final classifier on full training data for the bandit system
print(f"\n{'='*60}")
print("Training final classifier on full training data...")
final_user_classifier = DecisionTreeClassifier(random_state=42, max_depth=10)
final_user_classifier.fit(X_train_full, y_train_full)
print("Final classifier training completed!")


Training set size: 1600
Validation set size: 400

Training set label distribution:
label
user_2    570
user_1    565
user_3    465
Name: count, dtype: int64

Validation set label distribution:
label
user_2    142
user_1    142
user_3    116
Name: count, dtype: int64

User Classification Model Performance
Validation Accuracy: 0.8625

Classification Report on Validation Set:
              precision    recall  f1-score   support

      user_1       0.85      0.81      0.83       142
      user_2       0.93      0.85      0.89       142
      user_3       0.81      0.94      0.87       116

    accuracy                           0.86       400
   macro avg       0.86      0.87      0.86       400
weighted avg       0.87      0.86      0.86       400


Training final classifier on full training data...
Final classifier training completed!


# `Contextual Bandit`

## Reward Sampler Initialization

The sampler is initialized using the student's roll number `i`.
Rewards are obtained using `sampler.sample(j)`.


In [11]:
# Initialize the sampler with roll number
roll_number = 42
reward_sampler = sampler(roll_number)

print(f"Reward sampler initialized with roll number: {roll_number}")

# Define arm mapping as per the problem specification
# j values: 0-3 (User1), 4-7 (User2), 8-11 (User3)
# Categories: Entertainment, Education, Tech, Crime

news_categories = ['Entertainment', 'Education', 'Tech', 'Crime']
user_contexts = ['user_1', 'user_2', 'user_3']

# Create arm mapping dictionary
arm_mapping = {}
arm_index = 0
for user_idx, user in enumerate(user_contexts):
    for cat_idx, category in enumerate(news_categories):
        arm_mapping[arm_index] = {
            'user_context': user,
            'news_category': category,
            'context_idx': user_idx,
            'category_idx': cat_idx
        }
        arm_index += 1

# Display arm mapping
print(f"\n{'='*70}")
print("Arm Mapping Configuration")
print(f"{'='*70}")
print(f"{'Arm Index':<12} {'News Category':<15} {'User Context':<15}")
print(f"{'-'*70}")
for j in range(12):
    mapping = arm_mapping[j]
    print(f"{j:<12} {mapping['news_category']:<15} {mapping['user_context']:<15}")

# Helper function to get arm index from context and category
def get_arm_index(user_context, news_category):
    user_idx = user_contexts.index(user_context)
    cat_idx = news_categories.index(news_category)
    return user_idx * 4 + cat_idx

# Helper function to get context and category from arm index
def get_context_category(arm_index):
    return arm_mapping[arm_index]['user_context'], arm_mapping[arm_index]['news_category']

print(f"\n{'='*70}")
print("Arm mapping setup completed!")

Reward sampler initialized with roll number: 42

Arm Mapping Configuration
Arm Index    News Category   User Context   
----------------------------------------------------------------------
0            Entertainment   user_1         
1            Education       user_1         
2            Tech            user_1         
3            Crime           user_1         
4            Entertainment   user_2         
5            Education       user_2         
6            Tech            user_2         
7            Crime           user_2         
8            Entertainment   user_3         
9            Education       user_3         
10           Tech            user_3         
11           Crime           user_3         

Arm mapping setup completed!


## Arm Mapping

| Arm Index (j) | News Category | User Context |
|--------------|---------------|--------------|
| 0–3          | Entertainment, Education, Tech, Crime | User1 |
| 4–7          | Entertainment, Education, Tech, Crime | User2 |
| 8–11         | Entertainment, Education, Tech, Crime | User3 |

## Epsilon-Greedy Strategy

This section implements the epsilon-greedy contextual bandit algorithm.


In [12]:
class EpsilonGreedy:
    def __init__(self, n_contexts, n_arms_per_context, epsilon):
        self.n_contexts = n_contexts
        self.n_arms_per_context = n_arms_per_context
        self.epsilon = epsilon
        
        # For each context, maintain statistics for each arm
        self.Q = np.zeros((n_contexts, n_arms_per_context))  # Estimated rewards
        self.N = np.zeros((n_contexts, n_arms_per_context))  # Number of times each arm pulled
        self.total_reward = 0
        self.reward_history = []
        
    def select_arm(self, context_idx):
        # Epsilon-greedy selection for the given context
        if np.random.random() < self.epsilon:
            # Explore: random arm
            arm_idx = np.random.randint(0, self.n_arms_per_context)
        else:
            # Exploit: best arm for this context
            arm_idx = np.argmax(self.Q[context_idx])
        return arm_idx
    
    def update(self, context_idx, arm_idx, reward):
        # Update statistics
        self.N[context_idx, arm_idx] += 1
        n = self.N[context_idx, arm_idx]
        q = self.Q[context_idx, arm_idx]
        
        # Incremental update of estimated reward
        self.Q[context_idx, arm_idx] = q + (reward - q) / n
        
        self.total_reward += reward
        self.reward_history.append(reward)
    
    def get_expected_rewards(self):
        return self.Q.copy()
    
    def get_arm_counts(self):
        return self.N.copy()

print("Epsilon-Greedy class implementation completed!")
print("\nClass features:")
print("- Maintains Q values (estimated rewards) for each context-arm pair")
print("- Epsilon-greedy selection: explore with probability epsilon")
print("- Incremental reward estimation")

Epsilon-Greedy class implementation completed!

Class features:
- Maintains Q values (estimated rewards) for each context-arm pair
- Epsilon-greedy selection: explore with probability epsilon
- Incremental reward estimation


## Upper Confidence Bound (UCB)

This section implements the UCB strategy for contextual bandits.

In [13]:
class UCB:
    def __init__(self, n_contexts, n_arms_per_context, c):
        self.n_contexts = n_contexts
        self.n_arms_per_context = n_arms_per_context
        self.c = c  # Exploration parameter
        
        # For each context, maintain statistics for each arm
        self.Q = np.zeros((n_contexts, n_arms_per_context))  # Estimated rewards
        self.N = np.zeros((n_contexts, n_arms_per_context))  # Number of times each arm pulled
        self.total_reward = 0
        self.reward_history = []
        self.t = 0  # Total number of steps
        
    def select_arm(self, context_idx):
        # UCB selection for the given context
        ucb_values = np.zeros(self.n_arms_per_context)
        
        for arm_idx in range(self.n_arms_per_context):
            if self.N[context_idx, arm_idx] == 0:
                # If arm not pulled yet, give it infinite value to ensure exploration
                return arm_idx
            else:
                # UCB formula: Q(a) + c * sqrt(ln(t) / N(a))
                exploration_bonus = self.c * np.sqrt(np.log(self.t + 1) / self.N[context_idx, arm_idx])
                ucb_values[arm_idx] = self.Q[context_idx, arm_idx] + exploration_bonus
        
        return np.argmax(ucb_values)
    
    def update(self, context_idx, arm_idx, reward):
        # Update statistics
        self.N[context_idx, arm_idx] += 1
        n = self.N[context_idx, arm_idx]
        q = self.Q[context_idx, arm_idx]
        
        # Incremental update of estimated reward
        self.Q[context_idx, arm_idx] = q + (reward - q) / n
        
        self.total_reward += reward
        self.reward_history.append(reward)
        self.t += 1
    
    def get_expected_rewards(self):
        return self.Q.copy()
    
    def get_arm_counts(self):
        return self.N.copy()

print("UCB class implementation completed!")
print("\nClass features:")
print("- UCB selection with exploration bonus: Q(a) + c * sqrt(ln(t) / N(a))")
print("- Prioritizes arms with high uncertainty")
print("- Parameter c controls exploration intensity")


UCB class implementation completed!

Class features:
- UCB selection with exploration bonus: Q(a) + c * sqrt(ln(t) / N(a))
- Prioritizes arms with high uncertainty
- Parameter c controls exploration intensity


## SoftMax Strategy

This section implements the SoftMax strategy with temperature $ \tau = 1$.


In [ ]:
class SoftMax:
    def __init__(self, n_contexts, n_arms_per_context, tau):
        self.n_contexts = n_contexts
        self.n_arms_per_context = n_arms_per_context
        self.tau = tau  # Temperature parameter
        
        # For each context, maintain statistics for each arm
        self.Q = np.zeros((n_contexts, n_arms_per_context))  # Estimated rewards
        self.N = np.zeros((n_contexts, n_arms_per_context))  # Number of times each arm pulled
        self.total_reward = 0
        self.reward_history = []
        
    def select_arm(self, context_idx):
        # SoftMax selection for the given context
        # Handle case where all arms have zero Q values initially
        if np.all(self.N[context_idx] == 0):
            # Random selection initially
            return np.random.randint(0, self.n_arms_per_context)
        
        # SoftMax probabilities: P(a) = exp(Q(a)/tau) / sum(exp(Q(a)/tau))
        q_values = self.Q[context_idx]
        
        # Numerical stability: subtract max to avoid overflow
        q_shifted = q_values - np.max(q_values)
        exp_values = np.exp(q_shifted / self.tau)
        probabilities = exp_values / np.sum(exp_values)
        
        # Sample arm according to probabilities
        arm_idx = np.random.choice(self.n_arms_per_context, p=probabilities)
        return arm_idx
    
    def update(self, context_idx, arm_idx, reward):
        # Update statistics
        self.N[context_idx, arm_idx] += 1
        n = self.N[context_idx, arm_idx]
        q = self.Q[context_idx, arm_idx]
        
        # Incremental update of estimated reward
        self.Q[context_idx, arm_idx] = q + (reward - q) / n
        
        self.total_reward += reward
        self.reward_history.append(reward)
    
    def get_expected_rewards(self):
        return self.Q.copy()
    
    def get_arm_counts(self):
        return self.N.copy()

print("SoftMax class implementation completed!")
print("\nClass features:")
print("- Probabilistic arm selection based on Boltzmann distribution")
print("- Temperature tau = 1 (as specified)")
print("- Higher Q values get higher selection probability")
print("- Softer exploration compared to epsilon-greedy")

## Reinforcement Learning Simulation

We simulate the bandit algorithms for $T = 10,000$ steps and record rewards.

P.S.: Change $T$ value as and if required.


## Results and Analysis

This section presents:
- Average Reward vs Time
- Hyperparameter comparisons
- Observations and discussion


## Final Observations

- Comparison of Epsilon-Greedy, UCB, and SoftMax
- Effect of hyperparameters
- Strengths and limitations of each approach
